In [1]:
# --- Configuración de entorno ---

# Añade el directorio raíz al path para que Python encuentre tus módulos
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))  # Sube dos niveles hasta /src

# Importa configuraciones y librerías globales
from config import *
from utils import *

In [2]:
# Cargo un listado con las coordinadas de cada CP
ruta_csv_cps = os.path.join(DATA_INPUTS_DIR, "cps.csv")
df_cps = pd.read_csv(ruta_csv_cps, sep="\t", encoding="utf-8", header=None)
df_cps.sample(5)

,0,1,2,3,4,5,6,7,8,9,10,11
944,ES,11648,Espera,Andalucia,AN,Cádiz,CA,Espera,11017.0,36.8720,-5.8054,4
13991,ES,24339,Luengos,Castilla - Leon,CL,León,LE,Santas Martas,24160.0,42.4496,-5.3959,4
26841,ES,27259,Viladonga (Santiago),Galicia,GA,Lugo,LU,Castro de Rei,27010.0,43.1722,-7.4056,3
9727,ES,33879,Grandamuelle,Asturias,AS,Asturias,O,Tineo,33073.0,43.3577,-6.4077,3
24809,ES,15185,Xesteda (Santa Comba),Galicia,GA,A Coruña,C,Cerceda,15024.0,43.1983,-8.4367,3


In [3]:
# Filtro por CYL
df_cps = df_cps[df_cps[3] == "Castilla - Leon"]
df_cps.sample(5)

,0,1,2,3,4,5,6,7,8,9,10,11
13177,ES,9572,San Cibrian,Castilla - Leon,CL,Burgos,BU,Alfoz de Bricia,9011.0,42.9711,-3.7601,4
13768,ES,24225,Nava De Los Oteros,Castilla - Leon,CL,León,LE,Campo de Villavidel,24033.0,42.3935,-5.4652,4
14232,ES,24448,Toral De Merayo,Castilla - Leon,CL,León,LE,Ponferrada,24115.0,42.5243,-6.6355,4
13457,ES,34492,Rebolledo De La Torre,Castilla - Leon,CL,Burgos,BU,Rebolledo de la Torre,9306.0,42.6893,-4.2269,4
15996,ES,37775,Fresnedoso,Castilla - Leon,CL,Salamanca,SA,Fresnedoso,37133.0,40.4363,-5.7099,4


In [4]:
# Relleno con 0 para que tenga 5 caracteres
df_cps[1] = df_cps[1].astype(str)

# Añadir ceros a la izquierda si tiene menos de 5 dígitos
df_cps[1] = df_cps[1].str.zfill(5)

In [5]:
df_cps = df_cps.rename(columns={1: "CP", 7: "Loc",  9: "lat", 10: "lon"})
df_cps.head()

,0,CP,2,3,4,5,6,Loc,8,lat,lon,11
11765,ES,05001,Avila,Castilla - Leon,CL,Ávila,AV,Ávila,5019.0,40.6572,-4.6995,4
11766,ES,05002,Avila,Castilla - Leon,CL,Ávila,AV,Ávila,5019.0,40.6572,-4.6995,4
11767,ES,05003,Avila,Castilla - Leon,CL,Ávila,AV,Ávila,5019.0,40.6572,-4.6995,4
11768,ES,05004,Avila,Castilla - Leon,CL,Ávila,AV,Ávila,5019.0,40.6572,-4.6995,4
11769,ES,05005,Avila,Castilla - Leon,CL,Ávila,AV,Ávila,5019.0,40.6572,-4.6995,4


In [6]:
# Cargo el shp de CYL con los CUSEC para asignar cada CP a un CUSEC por su latitud y longitud
ruta_shp_secciones = os.path.join(DATA_OUTPUTS_DIR, "Shapefiles", "cyl_2022.shp")
gdf_secciones = gpd.read_file(ruta_shp_secciones).to_crs(epsg=25830)
gdf_secciones.head()

,CUSEC,NMUN,NPRO,Seccion_id,geometry
0,0500101001,Adanero,Ávila,1,"POLYGON ((365705.918 4536187.034, 365958.915 4..."
1,0500201001,"Adrada, La",Ávila,2,"POLYGON ((363065.743 4462346.46, 363062.106 44..."
2,0500201002,"Adrada, La",Ávila,3,"POLYGON ((361529.181 4469725.932, 361631.182 4..."
3,0500501001,Albornos,Ávila,4,"POLYGON ((343504.663 4523882.125, 343549.661 4..."
4,0500701001,Aldeanueva de Santa Cruz,Ávila,5,"POLYGON ((294940.799 4473589.074, 294982.799 4..."


In [7]:
from shapely.geometry import Point
import geopandas as gpd
import pandas as pd

# 1️⃣ Crear geometría de puntos desde lon/lat
geometry = [Point(xy) for xy in zip(df_cps["lon"], df_cps["lat"])]

gdf_cps = gpd.GeoDataFrame(
    df_cps,
    geometry=geometry,
    crs="EPSG:4326"   # coordenadas originales (lat/lon)
)

# 2️⃣ Reproyectar al CRS del shapefile de secciones (EPSG:25830)
gdf_cps = gdf_cps.to_crs(gdf_secciones.crs)

# 3️⃣ Join espacial → asignar CUSEC (relación 1 → N posible)
gdf_cps_cusec = gpd.sjoin(
    gdf_cps,
    gdf_secciones[["CUSEC", "geometry"]],
    how="left",
    predicate="within"
).drop(columns=["index_right"], errors="ignore")

print(f"🔎 Coincidencias iniciales: {len(gdf_cps_cusec)} (puede haber varios por CP)")

# 🚑 NUEVO PASO: Rescatar los puntos sin CUSEC con sjoin_nearest (distancia máxima 1 km)
sin_cusec = gdf_cps_cusec[gdf_cps_cusec["CUSEC"].isna()]
if len(sin_cusec) > 0:
    print(f"⚠️ Puntos sin CUSEC: {len(sin_cusec)} → buscando sección más cercana...")
    gdf_cps_nearest = gpd.sjoin_nearest(
        sin_cusec,
        gdf_secciones[["CUSEC", "geometry"]],
        how="left",
        max_distance=1000  # metros
    ).drop(columns=["index_right"], errors="ignore")
    
    # Solo actualiza si realmente existe la columna 'CUSEC'
    if "CUSEC" in gdf_cps_nearest.columns:
        gdf_cps_cusec.loc[gdf_cps_cusec["CUSEC"].isna(), "CUSEC"] = gdf_cps_nearest["CUSEC"].values
        print(f"✅ CUSEC asignado por proximidad a {len(gdf_cps_nearest)} puntos.")
    else:
        print("⚠️ No se encontró la columna 'CUSEC' en gdf_cps_nearest (posiblemente no hubo coincidencias cercanas).")

# 4️⃣ Consolidar → un solo registro por CP (coordenadas promedio y CUSEC principal)
df_cps_cusecs = (
    gdf_cps_cusec
    .groupby("CP", as_index=False)
    .agg({
        "lat": "mean",
        "lon": "mean",
        "CUSEC": "first"
    })
)

# 5️⃣ Asegurar formato limpio (texto con ceros)
df_cps_cusecs["CP"] = df_cps_cusecs["CP"].astype(str).str.zfill(5)
df_cps_cusecs["CUSEC"] = (
    df_cps_cusecs["CUSEC"]
    .fillna("")                              # evita NaN
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)    # elimina '.0' si viene de float
    .str.zfill(10)
)

# 6️⃣ Verificación final
muestras = df_cps_cusecs.sample(10)
validos = df_cps_cusecs["CUSEC"].str.match(r"^\d{10}$").sum()
total = len(df_cps_cusecs)

print(muestras)
print(f"\n✅ Total CPs únicos con CUSEC asignado: {validos} / {total}")
print(f"📊 Porcentaje de cobertura: {validos / total * 100:.2f}%")


🔎 Coincidencias iniciales: 6087 (puede haber varios por CP)
⚠️ Puntos sin CUSEC: 4 → buscando sección más cercana...
⚠️ No se encontró la columna 'CUSEC' en gdf_cps_nearest (posiblemente no hubo coincidencias cercanas).
         CP        lat       lon       CUSEC
148   05695  40.436500 -5.466000  0509701001
619   24513  42.627100 -6.807750  2420901002
1727  49009  41.506300 -5.744600  4927501003
628   24524  42.659000 -6.923733  2419801001
104   05420  40.291200 -4.583800  0524002001
667   24711  42.544775 -6.029350  2421401001
1618  47312  41.481400 -4.279400  4701201001
959   37181  40.954912 -5.559000  3706901001
223   09247  42.540483 -3.415250  0914301001
1667  47529  41.433100 -5.301900  4720401001

✅ Total CPs únicos con CUSEC asignado: 2020 / 2020
📊 Porcentaje de cobertura: 100.00%


In [8]:
# Ruta destino
ruta_csv_cps = os.path.join(DATA_OUTPUTS_DIR, "cps_cusecs.csv")

# Seleccionar solo las columnas relevantes
df_export = df_cps_cusecs[["CP", "lat", "lon", "CUSEC"]].copy()

# Guardar en CSV
df_export.to_csv(ruta_csv_cps, index=False, encoding="utf-8")

print(f"✅ Archivo guardado correctamente en: {ruta_csv_cps}")
print(f"💾 Total de registros: {len(df_export)}")
print(df_export.sample(5))

✅ Archivo guardado correctamente en: D:\MASTER EN CIENCIA DE DATOS\TFM\TrabajoFinal\ivst-tfm\data\outputs\cps_cusecs.csv
💾 Total de registros: 2020
         CP        lat       lon       CUSEC
1668  47530  41.481300 -5.284500  4715001001
1685  47673  42.009600 -5.472750  4919201001
1816  49191  41.408950 -5.690500  4914801001
777   24950  42.763900 -5.140300  2405601002
1111  37621  40.570662 -6.105075  3706101001
